<a href="https://colab.research.google.com/github/nicolaiberk/llm_ws/blob/main/notebooks/08_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building Your Own Measurement Tool

In this session, you are encouraged to play around with the available tools to play around with the available tools to develop your own measure.

Below, you can find the code to load existing datasets or use one of three endpoints for more computationally intensive tasks, but of course you are more than welcome to use your own data or any other method we've covered in the course and copy-paste the code from before!

## Datasets

### Public Datasets

Some publicly available datasets that you can use for your own analysis are:

#### Party Manifestos — Manifesto Project Corpus

Electoral programmes of parties in 60+ countries since 1945, human-annotated at the quasi-sentence level with 56 policy categories. Requires a free API key (register, then find it in your profile): https://manifesto-project.wzb.eu

In [ ]:
import requests
import pandas as pd

API_KEY = "YOUR_API_KEY"  # free: https://manifesto-project.wzb.eu -> profile
BASE = "https://manifesto-project.wzb.eu/api/v1"

# Party-level core dataset (one row per party x election)
raw = requests.get(
    f"{BASE}/get_core",
    params={"api_key": API_KEY, "key": "MPDS2024a"},
    timeout=120,
).json()
core = pd.DataFrame(raw[1:], columns=raw[0])
print(core.shape)

# Annotated full texts, e.g. the four largest German parties, 2021 election.
# Keys are "<party>_<yyyymm>"; look them up in `core` (columns: party, date).
keys = ["41320_202109", "41521_202109", "41113_202109", "41953_202109"]
resp = requests.get(
    f"{BASE}/texts_and_annotations",
    params={"api_key": API_KEY, "keys[]": keys, "version": "2024-1"},
    timeout=120,
).json()

rows = [
    {"manifesto_id": doc["key"], "text": it["text"], "cmp_code": it["cmp_code"]}
    for doc in resp["items"]
    for it in doc["items"]
]
manifestos = pd.DataFrame(rows)
print(manifestos.sample(5))

#### Policy Documents — MultiEURLEX (English subset)

65k EU laws from EUR-Lex in up to 23 languages, each annotated with multiple labels from the hierarchical EuroVoc taxonomy. Each EUROVOC label ID is associated with a label descriptor, e.g., [60, agri-foodstuffs], [6006, plant product], [1115, fruit]. 

Check out the details: https://huggingface.co/datasets/coastalcph/multi_eurlex.

In [ ]:
# Works with current datasets (>=3.0)
from datasets import load_dataset

BRANCH = "hf://datasets/coastalcph/multi_eurlex@refs/convert/parquet"
eurlex = load_dataset(
    "parquet",
    data_files={
        split: f"{BRANCH}/en/{split}/*.parquet"
        for split in ["train", "validation", "test"]
    },
)

In [ ]:
df = eurlex["train"].to_pandas()
print(df[["text", "labels"]].head())

#### Coded Policy Agendas — Comparative Agendas Project (CAP)

Human-coded policy documents (bills, laws, hearings, executive speeches, party platforms, media) from 20+ countries, all using one unified topic scheme (21 major topics, ~220 subtopics). Browse all datasets and codebooks:
https://www.comparativeagendas.net/datasets_codebooks

In [ ]:
import requests
import pandas as pd
import io

# 25,109 quasi-sentences from State of the Union speeches, 1946-2025
## Note: you can look for other data using the link above, copying their data URL
CAP_URL = "https://minio.la.utexas.edu/compagendas/datasetfiles/US-Exec_SOTU_2025.csv"
r = requests.get(CAP_URL, timeout=300)
r.raise_for_status()
cap = pd.read_csv(io.BytesIO(r.content), low_memory=False, encoding="latin-1")
print(cap.columns)

In [ ]:
cap[["description", "majortopic"]].head()

### Your own Dataset

You can also load your own dataset from a csv file in Dropbox with the following code:

In [ ]:
import pandas
mydata = pandas.read_csv("[YOUR DROPBOX LINK]&dl=1") # make sure to det dl=1

How to find the URL of your file in Dropbox:

<video controls src="vis/db_share.mov" title="Title"></video>

## Available Endpoints

I have set up three endpoints, which you can access via the API key provided via email. You can use these endpoints to generate embeddings, annotate with NLI, or use a hosted generative model.

### Fine-tuned Sentence Embeddings

The provided endpoint generates multilingual sentence embeddings using the sentence-transformers framework. You can use these embeddings to measure semantic similarity between sentences, which can be useful for various NLP tasks such as clustering, classification, and information retrieval.

You can use it as follows:

In [ ]:
import os
import requests

def query(payload):
	headers = {
		"Accept" : "application/json",
		"Authorization": "Bearer " + "[YOUR TOKEN]",
		"Content-Type": "application/json"
	}
	response = requests.post(
		"https://srrcjsqjtlh4ulxx.eu-west-1.aws.endpoints.huggingface.cloud",
		headers=headers,
		json=payload
	)
	return response.json()

output = query({
	"inputs": [
     "Berlin has about 1,000 Spaetis!", 
     "Berliners eat about 60 tons of Doner meat ever day!",
     "The red card was annulled after a call by president Trump."]
})

In [ ]:
## calculate cosine similarity
from numpy import dot
from numpy.linalg import norm


def cos_sim(a, b):
    cs = dot(a, b)/(norm(a)*norm(b))
    return(round(cs, 2))

print(cos_sim(output[0], output[1]))
print(cos_sim(output[1], output[2]))
print(cos_sim(output[0], output[2]))

### Natural Language Inference (NLI)

The provided endpoint generates zero-shot classification using natural-language inference (NLI). The [multilingual model](https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7) can perform NLI on 100 languages.

You can use it as follows:

In [ ]:
import os
import requests

def query(payload):
	headers = {
		"Accept" : "application/json",
		"Authorization": "Bearer " + "[YOUR TOKEN]",
		"Content-Type": "application/json"
	}
	response = requests.post(
		"https://us4vx4uhq2ap4fqh.eu-west-1.aws.endpoints.huggingface.cloud",
		headers=headers,
		json=payload
	)
	return response.json()

output = query({
	"inputs": "Die Klimakrise darf nicht weiter ignoriert werden!",
	"parameters": {
		"candidate_labels": ["climate", "economy", "migration"]
	}
}) 

print(output)

### Generative Model

You have access to an endpoint with a small version of the [APERTUS model](https://huggingface.co/swiss-ai/Apertus-8B-Instruct-2509) with 8 Billion parameters, which was instruction tuned. Apertus is a fully open-source language model, supporting over 1000 languages and long context. It uses only fully compliant and open training data.

You can use it as follows:

In [ ]:
# install the openai library first (uncomment to run)
# !pip install openai

In [ ]:
import requests

def query(payload):
	headers = {
		"Accept" : "application/json",
		"Authorization": f"Bearer " + HF_TOKEN,
		"Content-Type": "application/json"
	}
	response = requests.post(
		# note the endpoint URL: /v1/chat/completions is used for chat completion
		"https://pr04mvi6mfgikrkc.eu-west-1.aws.endpoints.huggingface.cloud/v1/chat/completions",
		headers=headers,
		 json={
        "model": "endpoint",  # ignored, the endpoint serves one model
        "messages": payload,
        "max_tokens": 50,
    	},
	)
	return response.json()

output = query([{
	"role": "user",
	"content": "Please annotate the following news snippet with the most likely topic: 'Die Klimakrise darf nicht weiter ignoriert werden!'",
	"parameters": {
		"temperature": 0,
		"max_new_tokens": 100
	}
}])


In [ ]:
print(output["choices"][0]["message"]["content"])

You can use this endpoint for structured calls as well - but please don't send your WHOLE data through the endpoint, but maybe restrict it to 20-30 examples at a time.

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.output import PromptedOutput

class CityLocation(BaseModel):
    city: str
    country: str

model = OpenAIChatModel(
    'swiss-ai/Apertus-8B-Instruct-2509',
    provider=OpenAIProvider(
        base_url='https://pr04mvi6mfgikrkc.eu-west-1.aws.endpoints.huggingface.cloud/v1/',
        api_key=os.environ['HF_TOKEN'],
    ),
)
agent = Agent(model, output_type=PromptedOutput(CityLocation))

result = agent.run_sync('Where were the olympics held in 2012?')
print(result.output)

# Your Turn

Start simple by exploring the data, maybe counting some words, and progress from there. Remember to look at the code from previous notebooks to see how to use the different tools.